# Forest Cover Type Prediction

In this notebook, Our Aim is to predict the forest cover type (the predominant kind of tree cover) from strictly cartographic variables (as opposed to remotely sensed data). The actual forest cover type for a given 30 x 30 meter cell was determined from US Forest Service (USFS) Region 2 Resource Information System data. Independent variables were then derived from data obtained from the US Geological Survey and USFS. 

**Id**: The unique identifier for each row/observation.<br>
**Elevation:** The elevation of the location in meters.<br>
**Aspect:** The aspect of the location in degrees azimuth.<br>
**Slope:** The slope of the location in degrees.<br>
**Horizontal_Distance_To_Hydrology:** The horizontal distance to the nearest surface water features in meters.<br>
**Vertical_Distance_To_Hydrology:** The vertical distance to the nearest surface water features in meters.<br>
**Horizontal_Distance_To_Roadways:** The horizontal distance to the nearest roadway in meters.<br>
**Hillshade_9am, Hillshade_Noon, Hillshade_3pm:** The hillshade index at the specified time of day.<br>
**Horizontal_Distance_To_Fire_Points:** The horizontal distance to the nearest wildfire ignition points in meters.<br>
**Wilderness_Area1, Wilderness_Area2, Wilderness_Area3, Wilderness_Area4:** Categorical columns indicating which of the four wilderness areas the location is in.<br>
**Soil_Type1 to Soil_Type40:** Categorical columns indicating which of the 40 soil types the location has.<br>
**Cover_Type:** The forest cover type, which is the target variable to be predicted.<br>

![](https://economictimes.indiatimes.com/thumb/msid-73039739,width-1200,height-900,resizemode-4,imgsize-366432/15.jpg?from=mdr)

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#read data and assign to variable
train = pd.read_csv("/kaggle/input/forest-cover-type-prediction/train.csv")
test = pd.read_csv("/kaggle/input/forest-cover-type-prediction/test.csv")
submission = pd.read_csv("/kaggle/input/forest-cover-type-prediction/sampleSubmission.csv")

In [ ]:
#checking first five rows
train.head()

In [ ]:
#checking for null values
train.isnull().sum()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# Histogram of Elevation
plt.figure(figsize=(8, 6))
sns.histplot(train['Elevation'])
plt.xlabel('Elevation')
plt.ylabel('Count')
plt.title('Distribution of Elevation')
plt.show()


In [ ]:
# Bar plot of Wilderness Areas
plt.figure(figsize=(8, 6))
train['Wilderness_Area1'].value_counts().plot(kind='bar')
plt.xlabel('Wilderness Area')
plt.ylabel('Count')
plt.title('Frequency of Wilderness Areas')
plt.show()

In [ ]:
# Heatmap of feature correlations
plt.figure(figsize=(12, 10))
corr = train.corr(numeric_only=True)
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
train.head()

In [ ]:
#splitting dependent and independent variables
x=train.drop(['Cover_Type'],axis=1)
y=train['Cover_Type']

In [ ]:
test_id=test['Id']

In [ ]:
#importing libraries
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

dt=DecisionTreeClassifier()
rf=RandomForestClassifier()
gb=GradientBoostingClassifier()



In [ ]:
#import standart scaler
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
x=sc.fit_transform(x)
test=sc.fit_transform(test)

In [ ]:
#training the model and predicting the test data
model=rf.fit(x,y)
y_pred=model.predict(test)

In [ ]:
import lightgbm as lgb
clf = lgb.LGBMClassifier()

params = {
    'num_leaves': 31,
    'objective': 'multiclass',
    'num_class': 7,
    'max_depth': -1,
    'learning_rate': 0.01,
    'min_child_samples': 20,
    'random_state': 0
}

lgb = lgb.LGBMClassifier(**params)


model=lgb.fit(x,y)
y_pred=model.predict(test)


In [ ]:
#submiting the predictions
submission=pd.DataFrame()
submission['Id']=test_id
submission['Cover_Type']=y_pred
submission.to_csv('submission.csv',index=False)
submission

In [ ]:
import pickle
pickle.dump(model,open('forestprediction.pkl','wb'))